# Week 4c: Ablation Study — CIFAR-10 CLIP (Dataset 3, N=1000)
**Project**: Quantum Kernels on CLIP/BERT Embeddings — ICLR Extension

**Same ablations as Week 4 / 4b, now on CIFAR-10 CLIP (vision modality).**
Every kernel matrix saved immediately to Drive after computation.

**Ablations:**
1. Feature maps — Angle vs IQP 1L (reused) vs IQP 2L (8 qubits)
2. Qubit count — 4 vs 8 (IQP 1L)
3. C parameter — sweep at N=200, apply best at N=1000

**Inputs:** `cifar10_embeddings.npz`, `cifar10_baseline_results.pkl`,
`cifar10_kernel_matrices.npz` (from Week 3c)

**Metric:** binary F1 (2 classes — airplane vs automobile)

**Runtime:** ~4 hrs (2 new kernel matrices + 4-qubit run)

**This completes the ablation study across all three datasets.**

In [ ]:
!pip install pennylane pennylane-lightning --quiet
import pennylane as qml
import numpy as np
import time, json, pickle, os, shutil, warnings
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
warnings.filterwarnings('ignore')
print(f"PennyLane {qml.__version__}")
dev4 = qml.device('lightning.qubit', wires=4)
dev8 = qml.device('lightning.qubit', wires=8)

---
## Step 0: Storage + helpers

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")

def _persist_and_download(path):
    if os.path.abspath(SAVE_DIR) != os.getcwd():
        shutil.copy(path, os.path.join(SAVE_DIR, os.path.basename(path)))
        print(f"  saved -> {SAVE_DIR}/{os.path.basename(path)}")
    try:
        from google.colab import files
        files.download(path)
        print(f"  download -> {os.path.basename(path)}")
    except Exception as e:
        print(f"  (download skipped: {type(e).__name__} — saved above)")

def checkpoint(filename, save_fn):
    save_fn(filename); _persist_and_download(filename)

def save_figure(fig, name):
    for ext in ('png', 'pdf'):
        p = f"{name}.{ext}"
        fig.savefig(p, bbox_inches='tight', dpi=150)
        _persist_and_download(p)

def _find(fname):
    dp = os.path.join(SAVE_DIR, fname)
    return dp if os.path.exists(dp) else fname

print("Helpers ready.")

---
## Step 1: Load data + Week 3c kernel matrix

In [ ]:
data  = np.load(_find('cifar10_embeddings.npz'))
wk3c  = np.load(_find('cifar10_kernel_matrices.npz'))

X_tr_full = data['X_tr_subset'].astype(np.float64)  # (1000, 8)
X_te_full = data['X_te_subset'].astype(np.float64)  # (200, 8)
y_tr_full = data['y_tr_subset']
y_te_full = data['y_te_subset']

K_wk3c_train = wk3c['K_train_quantum']  # 1000x1000 IQP 1L
K_wk3c_test  = wk3c['K_test_quantum']   # 200x1000

with open(_find('cifar10_baseline_results.pkl'), 'rb') as f:
    baselines = pickle.load(f)

N_ABL = 1000
CIFAR_CLASSES = {0: 'airplane', 1: 'automobile'}
X_tr, y_tr = X_tr_full[:N_ABL], y_tr_full[:N_ABL]
X_te, y_te = X_te_full, y_te_full

print(f"Train {X_tr.shape}  Test {X_te.shape}")
print(f"K_wk3c_train {K_wk3c_train.shape}  K_wk3c_test {K_wk3c_test.shape}")
assert K_wk3c_train.shape == (1000,1000), "Week 3c matrix not N=1000"

best_baseline_f1 = max(v['f1'] for v in baselines.values())
linear_f1 = baselines.get('Linear', {}).get('f1', 0)
print(f"\nBaselines (CLIP, N=1000):")
print(f"  {'Kernel':<14} {'Acc':>8} {'F1':>8}")
for k, v in baselines.items():
    print(f"  {k:<14} {v['accuracy']:>8.4f} {v['f1']:>8.4f}")

---
## Step 2: Feature maps

In [ ]:
def angle_map(x, wires):
    for i, w in enumerate(wires): qml.RY(x[i], wires=w)

def iqp_1layer(x, wires):
    n = len(wires)
    for w in wires: qml.Hadamard(wires=w)
    for i, w in enumerate(wires): qml.RZ(x[i], wires=w)
    for i in range(n-1):
        qml.CNOT(wires=[wires[i], wires[i+1]])
        qml.RZ(x[i]*x[i+1], wires=wires[i+1])
        qml.CNOT(wires=[wires[i], wires[i+1]])

def iqp_2layer(x, wires):
    for _ in range(2): iqp_1layer(x, wires)

def make_kernel(fmap, device, n_wires):
    @qml.qnode(device)
    def circ(x1, x2):
        fmap(x1, wires=range(n_wires))
        qml.adjoint(fmap)(x2, wires=range(n_wires))
        return qml.probs(wires=range(n_wires))
    return lambda a, b: float(circ(a, b)[0])

k_angle   = make_kernel(angle_map,  dev8, 8)
k_iqp1    = make_kernel(iqp_1layer, dev8, 8)
k_iqp2    = make_kernel(iqp_2layer, dev8, 8)
k_iqp1_4q = make_kernel(iqp_1layer, dev4, 4)

print("Sanity (K(x,x) must be 1.0):")
for nm, kf, x in [('Angle',k_angle,X_tr[0]),('IQP1L',k_iqp1,X_tr[0]),
                  ('IQP2L',k_iqp2,X_tr[0]),('IQP1L-4q',k_iqp1_4q,X_tr[0,:4])]:
    print(f"  {nm:<12} {kf(x,x):.6f}")

---
## Ablation 1: Feature maps (8 qubits, N=1000)
IQP 1L reused from Week 3c. Angle and IQP 2L each ~83 min.

In [ ]:
fm_results = {}

# IQP 1L — reuse Week 3c
y_w3c = SVC(kernel='precomputed', C=1.0).fit(K_wk3c_train, y_tr).predict(K_wk3c_test)
fm_results['IQP 1L'] = {
    'accuracy': float(accuracy_score(y_te, y_w3c)),
    'f1': float(f1_score(y_te, y_w3c)),
    'reused': True
}
print(f"IQP 1L (reused): acc={fm_results['IQP 1L']['accuracy']:.4f}  f1={fm_results['IQP 1L']['f1']:.4f}")

# Angle and IQP 2L — compute + save immediately
for fname, kfn, tag in [('Angle', k_angle, 'angle'), ('IQP 2L', k_iqp2, 'iqp2l')]:
    print(f"\nComputing {fname} (N={N_ABL}, ~83 min)...")
    t0 = time.time()
    Ktr = qml.kernels.kernel_matrix(X_tr, X_tr, kfn)
    Kte = qml.kernels.kernel_matrix(X_te, X_tr, kfn)
    print(f"  done {(time.time()-t0)/60:.1f} min — saving immediately:")
    checkpoint(f'cifar10_kernel_{tag}_n1000.npz',
               lambda p, a=Ktr, b=Kte: np.savez(p, K_train=a, K_test=b))
    y_p = SVC(kernel='precomputed', C=1.0).fit(Ktr, y_tr).predict(Kte)
    fm_results[fname] = {
        'accuracy': float(accuracy_score(y_te, y_p)),
        'f1': float(f1_score(y_te, y_p))
    }
    print(f"  {fname}: acc={fm_results[fname]['accuracy']:.4f}  f1={fm_results[fname]['f1']:.4f}")

print(f"\n{'Feature Map':<12} {'Acc':>8} {'F1':>8}")
print("-"*30)
for n, r in fm_results.items():
    print(f"{n:<12} {r['accuracy']:>8.4f} {r['f1']:>8.4f}")

---
## Ablation 2: Qubit count (IQP 1L, N=1000)

In [ ]:
qubit_results = {
    8: {'accuracy': fm_results['IQP 1L']['accuracy'],
        'f1': fm_results['IQP 1L']['f1'], 'reused': True}
}

print(f"Computing IQP 1L 4-qubit (N={N_ABL}, ~83 min)...")
t0 = time.time()
K4_tr = qml.kernels.kernel_matrix(X_tr[:, :4], X_tr[:, :4], k_iqp1_4q)
K4_te = qml.kernels.kernel_matrix(X_te[:, :4], X_tr[:, :4], k_iqp1_4q)
print(f"  done {(time.time()-t0)/60:.1f} min — saving:")
checkpoint('cifar10_kernel_iqp1l_4q_n1000.npz',
           lambda p: np.savez(p, K_train=K4_tr, K_test=K4_te))
y_p4 = SVC(kernel='precomputed', C=1.0).fit(K4_tr, y_tr).predict(K4_te)
qubit_results[4] = {
    'accuracy': float(accuracy_score(y_te, y_p4)),
    'f1': float(f1_score(y_te, y_p4))
}
print(f"  4q: acc={qubit_results[4]['accuracy']:.4f}  f1={qubit_results[4]['f1']:.4f}")

print(f"\n{'Qubits':>7} {'Acc':>8} {'F1':>8}")
for n_q in sorted(qubit_results):
    r = qubit_results[n_q]
    print(f"{n_q:>7} {r['accuracy']:>8.4f} {r['f1']:>8.4f}")
print(f"{'12':>7}   (re-run Week 2c with n_components=12)")

---
## Ablation 3: C parameter (sweep N=200, apply best at N=1000)

In [ ]:
print("C sweep at N=200...")
t0 = time.time()
Kc_tr = qml.kernels.kernel_matrix(X_tr[:200], X_tr[:200], k_iqp1)
Kc_te = qml.kernels.kernel_matrix(X_te, X_tr[:200], k_iqp1)
print(f"  done in {time.time()-t0:.0f}s")

C_values  = [0.1, 0.5, 1.0, 5.0, 10.0]
c_results = {}
print(f"\n{'C':>6} {'Acc':>8} {'F1':>8}")
print("-"*24)
for C in C_values:
    yp = SVC(kernel='precomputed', C=C).fit(Kc_tr, y_tr[:200]).predict(Kc_te)
    c_results[C] = {'accuracy': float(accuracy_score(y_te, yp)),
                    'f1': float(f1_score(y_te, yp))}
    print(f"{C:>6.1f} {c_results[C]['accuracy']:>8.4f} {c_results[C]['f1']:>8.4f}")

best_C = max(c_results, key=lambda c: c_results[c]['f1'])
print(f"\nBest C = {best_C} (F1={c_results[best_C]['f1']:.4f})")

if best_C == 1.0:
    print("C=1.0 already used — no re-run needed.")
else:
    yb = SVC(kernel='precomputed', C=best_C).fit(K_wk3c_train, y_tr).predict(K_wk3c_test)
    print(f"IQP 1L @ C={best_C}, N=1000: "
          f"acc={accuracy_score(y_te,yb):.4f}  f1={f1_score(y_te,yb):.4f}")

---
## Paper Table: CIFAR-10 ablation results

In [ ]:
print("="*66)
print(f"TABLE — CIFAR-10 binary (vision, CLIP, 8-dim, N={N_ABL})")
print("="*66)
print(f"{'Method':<22} {'Qubits':>7} {'N':>6} {'Acc':>8} {'F1':>8} {'vsLinear':>9}")
print("-"*66)

for name, v in baselines.items():
    gap = v['f1'] - linear_f1
    print(f"  {'Classical — '+name:<20} {'—':>7} {'1000':>6} "
          f"{v['accuracy']:>8.4f} {v['f1']:>8.4f} {gap:>+9.1%}")
print()
for fname, r in fm_results.items():
    gap  = r['f1'] - linear_f1
    note = ' *' if fname=='IQP 1L' else ''
    print(f"  {'Quantum — '+fname:<20} {'8':>7} {'1000':>6} "
          f"{r['accuracy']:>8.4f} {r['f1']:>8.4f} {gap:>+9.1%}{note}")
print()
for n_q in sorted(qubit_results):
    r = qubit_results[n_q]
    gap = r['f1'] - linear_f1
    print(f"  {'Quantum — IQP 1L':<20} {n_q:>7} {'1000':>6} "
          f"{r['accuracy']:>8.4f} {r['f1']:>8.4f} {gap:>+9.1%}")
print(f"  {'Quantum — IQP 1L':<20} {'12':>7} {'—':>6}  {'TBD':>8}")
print("="*66)
print("* reused from Week 3c")

---
## Figures

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

fm_names = list(fm_results.keys())
fm_f1s   = [fm_results[n]['f1'] for n in fm_names]
axes[0].bar(fm_names, fm_f1s, color=['#4D9DE0','#E15554','#3BB273'], width=0.5, edgecolor='white')
axes[0].axhline(best_baseline_f1, color='gray', ls='--', alpha=0.6, label='Best classical baseline')
axes[0].set_ylim(0.4, 1.0); axes[0].set_title(f'Feature maps (8q, N={N_ABL})')
axes[0].set_ylabel('F1'); axes[0].legend(fontsize=9)
for b, v in zip(axes[0].patches, fm_f1s):
    axes[0].text(b.get_x()+b.get_width()/2, v+0.004, f'{v:.3f}', ha='center', fontsize=10, fontweight='500')

q_labels = [f'{n}q' for n in sorted(qubit_results)]+['12q*']
q_f1s    = [qubit_results[n]['f1'] for n in sorted(qubit_results)]+[None]
bars2 = axes[1].bar(q_labels, [v if v else 0 for v in q_f1s],
                    color=['#4D9DE0','#E15554','#CCCCCC'], width=0.5, edgecolor='white')
axes[1].axhline(best_baseline_f1, color='gray', ls='--', alpha=0.6, label='Best classical baseline')
axes[1].set_ylim(0.4, 1.0); axes[1].set_title(f'Qubit count (IQP1L, N={N_ABL})')
axes[1].set_ylabel('F1'); axes[1].legend(fontsize=9)
for b, v in zip(bars2, q_f1s):
    lbl = f'{v:.3f}' if v else 'TBD'
    axes[1].text(b.get_x()+b.get_width()/2, b.get_height()+0.004, lbl, ha='center', fontsize=10, fontweight='500')

C_vals = list(c_results.keys())
c_f1s  = [c_results[c]['f1'] for c in C_vals]
bi = c_f1s.index(max(c_f1s))
axes[2].bar([str(c) for c in C_vals], c_f1s,
            color=['#E15554' if i==bi else '#4D9DE0' for i in range(len(C_vals))],
            width=0.6, edgecolor='white')
axes[2].axhline(best_baseline_f1, color='gray', ls='--', alpha=0.6, label='Best classical baseline')
axes[2].set_ylim(0.4, 1.0); axes[2].set_title('C sweep (N=200 heuristic)')
axes[2].set_xlabel('C'); axes[2].set_ylabel('F1'); axes[2].legend(fontsize=9)
for b, v in zip(axes[2].patches, c_f1s):
    axes[2].text(b.get_x()+b.get_width()/2, v+0.004, f'{v:.3f}', ha='center', fontsize=10, fontweight='500')

plt.suptitle(f'Week 4c Ablations — CIFAR-10 CLIP (vision), N={N_ABL}', y=1.02)
plt.tight_layout(); plt.show()
save_figure(fig, 'cifar10_fig3_ablations')

---
## Cross-dataset ablation comparison (all 3 datasets)

In [ ]:
# Load Week 4 and 4b ablation results for the full cross-dataset ablation picture
sst2_abl, ag_abl = None, None
try:
    with open(_find('results_week4.json')) as f: sst2_abl = json.load(f)
except: pass
try:
    with open(_find('ag_news_results_week4b.json')) as f: ag_abl = json.load(f)
except: pass

if sst2_abl and ag_abl:
    # Feature map effect across 3 datasets
    fmaps = ['Angle', 'IQP 1L', 'IQP 2L']
    def get_fm(res, key):
        fm = res.get('feature_map_ablation', {})
        # handle both 'f1' and 'f1_macro' keys
        v = fm.get(key, {})
        return v.get('f1', v.get('f1_macro', 0))

    sst2_fm  = [get_fm(sst2_abl, k) for k in fmaps]
    ag_fm    = [get_fm(ag_abl, k) for k in fmaps]
    cifar_fm = [fm_results[k]['f1'] for k in fmaps]

    fig2, ax = plt.subplots(figsize=(11, 5))
    x = np.arange(len(fmaps)); w = 0.25
    ax.bar(x-w, sst2_fm,  w, label='SST-2 (BERT)',    color='#5B8DB8', edgecolor='white')
    ax.bar(x,   ag_fm,    w, label='AG News (BERT)',  color='#2D9C4A', edgecolor='white')
    ax.bar(x+w, cifar_fm, w, label='CIFAR-10 (CLIP)', color='#D95F02', edgecolor='white')
    ax.set_xticks(x); ax.set_xticklabels(fmaps, fontsize=11)
    ax.set_ylim(0.4, 0.9); ax.set_ylabel('F1')
    ax.set_title('Feature map ablation across 3 datasets — does the same circuit win everywhere?')
    ax.legend(); plt.tight_layout(); plt.show()
    save_figure(fig2, 'cross_dataset_feature_map_ablation')
    print("Cross-dataset ablation figure saved.")
else:
    print("Week 4 / 4b results not found — skipping cross-dataset ablation figure.")

---
## Save results

In [ ]:
all_results = {
    'dataset': 'cifar10_binary',
    'modality': 'vision',
    'N_ABL': N_ABL,
    'feature_map_ablation': fm_results,
    'qubit_ablation': {str(k): v for k, v in qubit_results.items()},
    'c_sweep': {str(k): v for k, v in c_results.items()},
    'best_C': float(best_C),
    'classical_baselines': baselines,
}
checkpoint('cifar10_results_week4c.json',
           lambda p: json.dump(all_results, open(p,'w'), indent=2))

print("\n=== WEEK 4c DONE — ALL ABLATIONS COMPLETE ===")
print("You now have ablation studies on all 3 datasets:")
print("  SST-2 (Week 4), AG News (Week 4b), CIFAR-10 (Week 4c)")
print(f"\nAll CIFAR-10 files in Drive:")
cf_files = [f for f in sorted(os.listdir(SAVE_DIR)) if 'cifar10' in f]
for f in cf_files:
    print(f"  {f}")

---
## Week 4c complete — All Ablations Done

### Saved + downloaded:
- `cifar10_kernel_angle_n1000.npz` — saved immediately after Angle map
- `cifar10_kernel_iqp2l_n1000.npz` — saved immediately after IQP 2L
- `cifar10_kernel_iqp1l_4q_n1000.npz` — saved immediately after 4-qubit run
- `cifar10_fig3_ablations.png/.pdf`
- `cross_dataset_feature_map_ablation.png/.pdf` — all 3 datasets compared
- `cifar10_results_week4c.json`

### The complete experimental matrix is now done:
Three datasets, three feature maps, two qubit counts, C sweeps, all at N=1000.
Across two modalities (BERT text, CLIP vision) and three task types
(binary sentiment, 4-class topic, binary vision).

### What the cross-dataset ablation tells you:
If IQP 2L beats IQP 1L on all three datasets, the feature map ranking is robust.
If the qubit-count effect is consistent, that generalises too. These consistency
checks are what turn three separate results into one coherent finding.

### Remaining for the paper:
- Week 5: KTA analysis across all 3 datasets (your predictor thesis), t-SNE, significance tests
- Week 6: writing

The experimental phase is essentially complete. Time to analyse and write.